# 03 · Logistic regression, gradient checking, and Newton's method

**Goal:** a classifier with no closed form, so you need an optimizer. You will also learn **gradient checking**, which catches most backprop bugs later on.

For labels $y_i\in\{0,1\}$ and $p_i=\sigma(x_i^\top w)$, $\sigma(z)=1/(1+e^{-z})$:

$$L(w) = -\frac1n\sum_i \big[y_i\log p_i+(1-y_i)\log(1-p_i)\big],\qquad
\nabla L=\frac1n X^\top(p-y),\qquad
\nabla^2L=\frac1n X^\top \mathrm{diag}\big(p_i(1-p_i)\big)X .$$

The Hessian is PSD, so $L$ is **convex**: any local minimum is global.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def loss(w, X, y):
    p = sigmoid(X @ w)
    eps = 1e-12
    return -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))

def grad(w, X, y):
    return X.T @ (sigmoid(X @ w) - y) / len(y)

def hess(w, X, y):
    p = sigmoid(X @ w)
    return X.T @ (X * (p * (1 - p))[:, None]) / len(y)

# two overlapping Gaussian blobs
m = 200
X0 = rng.normal(size=(m, 2)) + np.array([-1.2, -1.2])
X1 = rng.normal(size=(m, 2)) + np.array([1.2, 1.2])
X = np.c_[np.ones(2 * m), np.r_[X0, X1]]
y = np.r_[np.zeros(m), np.ones(m)]

## Gradient checking

Compare the analytic gradient with the centered finite difference $\frac{L(w+he_i)-L(w-he_i)}{2h}$. Relative error around $10^{-8}$ or better means the gradient is right.

In [ ]:
def num_grad(f, w, h=1e-6):
    g = np.zeros_like(w)
    for i in range(len(w)):
        e = np.zeros_like(w); e[i] = h
        g[i] = (f(w + e) - f(w - e)) / (2 * h)
    return g

w_test = rng.normal(size=3)
g_an = grad(w_test, X, y)
g_nm = num_grad(lambda w: loss(w, X, y), w_test)
rel = np.linalg.norm(g_an - g_nm) / np.linalg.norm(g_an + g_nm)
print("relative error:", rel)
assert rel < 1e-6

# Hessian check: differentiate the gradient
H_an = hess(w_test, X, y)
H_nm = np.column_stack([num_grad(lambda w, i=i: grad(w, X, y)[i], w_test) for i in range(3)])
assert np.allclose(H_an, H_nm, atol=1e-6)

## Gradient descent vs Newton

Newton's method uses curvature: $w\leftarrow w-H^{-1}\nabla L$. It converges **quadratically** near the optimum, at the price of forming and solving with $H$ each step ($O(d^3)$). For logistic regression this is also known as IRLS.

In [ ]:
def run_gd(eta=0.5, steps=500):
    w = np.zeros(3); h = []
    for _ in range(steps):
        h.append(loss(w, X, y)); w = w - eta * grad(w, X, y)
    return w, np.array(h)

def run_newton(steps=10):
    w = np.zeros(3); h = []
    for _ in range(steps):
        h.append(loss(w, X, y))
        w = w - np.linalg.solve(hess(w, X, y), grad(w, X, y))
    return w, np.array(h)

w_gd, h_gd = run_gd()
w_nt, h_nt = run_newton()
print("GD     final loss:", h_gd[-1].round(6), " w =", w_gd.round(3))
print("Newton final loss:", h_nt[-1].round(6), " w =", w_nt.round(3))
assert np.allclose(grad(w_nt, X, y), 0, atol=1e-8)
assert abs(h_gd[-1] - h_nt[-1]) < 1e-3

acc = np.mean((sigmoid(X @ w_nt) > 0.5) == y)
print("training accuracy:", acc)
assert acc > 0.9

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].semilogy(h_gd - h_nt[-1] + 1e-16, label="GD")
axs[0].semilogy(h_nt - h_nt[-1] + 1e-16, "o-", label="Newton")
axs[0].set_xlim(0, 30); axs[0].legend(); axs[0].set_title("suboptimality")
axs[1].scatter(X0[:, 0], X0[:, 1], s=8); axs[1].scatter(X1[:, 0], X1[:, 1], s=8)
xs = np.linspace(-4, 4, 2)
axs[1].plot(xs, -(w_nt[0] + w_nt[1] * xs) / w_nt[2], "k"); axs[1].set_title("decision boundary")
plt.show()

## Exercises

1. Add L2 regularization $\tfrac{\lambda}{2}\|w_{1:}\|^2$ to loss, gradient *and* Hessian, then re-run the gradient check.
2. Move the blobs far apart so the data are linearly separable. What happens to $\|w\|$ under GD and to Newton's Hessian? Explain.
3. Extend to softmax regression with $K$ classes; derive $\nabla L = \tfrac1n X^\top(P-Y)$.